# Landmark SLAM: Q Mat Visuals

Compute and sanity-check the observation matrix `Q_n` for landmark-based SLAM.

In [ ]:
import sys
import os
import subprocess
from pathlib import Path

# Ensure project import path
PROJECT_ROOT = Path('/global/home/hpc5656/SLAM')
sys.path.append(str(PROJECT_ROOT))
# Set working directory so relative paths (e.g., src/config/*.yaml) resolve
os.chdir(str(PROJECT_ROOT))
print('CWD:', Path.cwd())

%matplotlib inline

# Enable autoreload for development (automatically reloads modules when they change)
%load_ext autoreload
%autoreload 2

# CuPy is required - ensure CUDA_PATH and LD_LIBRARY_PATH are set
# This allows the notebook to work even if Jupyter wasn't started with modules loaded
if "CUDA_PATH" not in os.environ:
    print("CUDA_PATH not set, attempting to load modules...")
    try:
        result = subprocess.run(
            'module load cuda/12.2 && env',
            shell=True,
            executable='/bin/bash',
            capture_output=True,
            text=True,
            timeout=10
        )
        if result.returncode == 0:
            for line in result.stdout.split('\n'):
                if '=' in line:
                    key, value = line.split('=', 1)
                    os.environ[key] = value
            print(f"✓ Modules loaded. CUDA_PATH: {os.environ.get('CUDA_PATH', 'Not set')}")
        else:
            print(f"⚠ Could not load modules. Error: {result.stderr}")
            raise RuntimeError(
                "CUDA_PATH not set and could not load modules. "
                "Please run: module load cuda/12.2 before starting Jupyter."
            )
    except Exception as e:
        print(f"✗ Could not load modules: {e}")
        raise RuntimeError(
            f"Failed to load CUDA modules: {e}\n"
            "Please ensure CUDA is loaded before starting Jupyter:\n"
            "  module load cuda/12.2"
        ) from e

# Ensure LD_LIBRARY_PATH includes CUDA library directory for NVRTC (libnvrtc.so.12)
# This is required for CuPy to compile kernels at runtime
# Note: Setting this BEFORE importing CuPy is critical
cuda_path = os.environ.get('CUDA_PATH')
libnvrtc_path = None

if cuda_path:
    # Check both standard location and Compute Canada's targets/x86_64-linux/lib location
    cuda_lib_paths = [
        os.path.join(cuda_path, 'lib64'),  # Standard location
        os.path.join(cuda_path, 'targets', 'x86_64-linux', 'lib'),  # Compute Canada location
    ]
    
    current_ld_path = os.environ.get('LD_LIBRARY_PATH', '')
    ld_paths = current_ld_path.split(':') if current_ld_path else []
    paths_added = []
    
    # Find which paths exist and add them to LD_LIBRARY_PATH
    for cuda_lib_path in cuda_lib_paths:
        if os.path.exists(cuda_lib_path):
            if cuda_lib_path not in ld_paths:
                paths_added.append(cuda_lib_path)
                ld_paths.insert(0, cuda_lib_path)  # Prepend for priority
    
    if paths_added:
        os.environ['LD_LIBRARY_PATH'] = ':'.join(ld_paths)
        print(f"✓ Updated LD_LIBRARY_PATH to include: {', '.join(paths_added)}")
    else:
        # Check if paths were already included
        found_paths = [p for p in cuda_lib_paths if p in ld_paths]
        if found_paths:
            print(f"✓ LD_LIBRARY_PATH already includes CUDA libraries: {', '.join(found_paths)}")
    
    # Find libnvrtc.so.12 and preload it using ctypes
    # This ensures CuPy can find it even if LD_LIBRARY_PATH isn't fully respected
    for cuda_lib_path in cuda_lib_paths:
        if os.path.exists(cuda_lib_path):
            potential_libnvrtc = os.path.join(cuda_lib_path, 'libnvrtc.so.12')
            if os.path.exists(potential_libnvrtc):
                libnvrtc_path = potential_libnvrtc
                print(f"✓ Found libnvrtc.so.12 at: {libnvrtc_path}")
                
                # Preload the library using ctypes so CuPy can find it
                # Use RTLD_GLOBAL to make symbols available to other libraries
                try:
                    import ctypes
                    # Try multiple loading strategies
                    try:
                        # Strategy 1: Load with full path and RTLD_GLOBAL
                        lib = ctypes.CDLL(libnvrtc_path, mode=ctypes.RTLD_GLOBAL)
                        print(f"✓ Preloaded libnvrtc.so.12 using ctypes (RTLD_GLOBAL)")
                    except Exception as e1:
                        # Strategy 2: Try without RTLD_GLOBAL
                        try:
                            lib = ctypes.CDLL(libnvrtc_path)
                            print(f"✓ Preloaded libnvrtc.so.12 using ctypes (standard)")
                        except Exception as e2:
                            raise e1 from e2
                except Exception as e:
                    print(f"⚠ Warning: Could not preload libnvrtc.so.12: {e}")
                    print(f"  CuPy may still work if LD_LIBRARY_PATH is set correctly")
                    print(f"  You may need to restart the Jupyter kernel with:")
                    print(f"    export LD_LIBRARY_PATH={os.path.dirname(libnvrtc_path)}:$LD_LIBRARY_PATH")
                
                # Verify that ctypes can find the library by name (as CuPy will try)
                try:
                    import ctypes.util
                    found_lib = ctypes.util.find_library('nvrtc')
                    if found_lib:
                        print(f"✓ ctypes.util.find_library('nvrtc') found: {found_lib}")
                    else:
                        print(f"⚠ ctypes.util.find_library('nvrtc') returned None")
                        print(f"  This may cause issues. Try loading by name:")
                        try:
                            test_lib = ctypes.CDLL('libnvrtc.so.12')
                            print(f"✓ Successfully loaded libnvrtc.so.12 by name")
                        except Exception as name_err:
                            print(f"✗ Failed to load libnvrtc.so.12 by name: {name_err}")
                            print(f"  You MUST restart the Jupyter kernel with LD_LIBRARY_PATH set")
                except Exception as diag_err:
                    print(f"⚠ Could not run diagnostics: {diag_err}")
                break
    
    if not libnvrtc_path:
        # Try to find any version of libnvrtc.so
        import glob
        for cuda_lib_path in cuda_lib_paths:
            if os.path.exists(cuda_lib_path):
                nvrtc_files = glob.glob(os.path.join(cuda_lib_path, 'libnvrtc.so*'))
                if nvrtc_files:
                    # Try to use the most specific version
                    nvrtc_files.sort(reverse=True)  # Prefer .so.12.2.140 over .so.12 over .so
                    potential_lib = nvrtc_files[0]
                    print(f"⚠ libnvrtc.so.12 not found, but found: {nvrtc_files}")
                    print(f"  Attempting to use: {potential_lib}")
                    try:
                        import ctypes
                        ctypes.CDLL(potential_lib, mode=ctypes.RTLD_GLOBAL)
                        print(f"✓ Preloaded {potential_lib} using ctypes")
                        libnvrtc_path = potential_lib
                    except Exception as e:
                        print(f"⚠ Could not preload {potential_lib}: {e}")
                    break
        else:
            print(f"⚠ Warning: libnvrtc.so.12 not found in any CUDA library directory")
            print(f"  This may cause CuPy kernel compilation to fail")
else:
    print("⚠ CUDA_PATH not set, cannot configure LD_LIBRARY_PATH")

# Use CuPy backend for GPU acceleration (falls back to NumPy if not available)
from src.utils.array_backend import np, random, is_cupy
from src.belief_quantized.belief_mdp_n import BeliefMDP_n_SLAM
import numpy as np_cpu  # Import NumPy explicitly for CPU operations
from src.classes.mapping import OrderedLandmarkMap
from src.classes.model import RangeBearingSensor, DoubleIntegratorModel

print("✓ All imports successful")
if is_cupy:
    print("Using backend: CuPy (GPU)")
else:
    print("Using backend: NumPy (CPU)")

In [ ]:
# Landmark map setup
workspace_min = 0.0
workspace_max = 10.0

# Quantized landmark positions: X_n (n=3 -> 9 possible locations)
quantization_level = 3
cell_size = (workspace_max - workspace_min) / quantization_level
coords = workspace_min + cell_size * (np.arange(quantization_level) + 0.5)
xx, yy = np.meshgrid(coords, coords)
landmark_positions = np.stack([xx.ravel(), yy.ravel()], axis=1)

_map = OrderedLandmarkMap(
    x_min=workspace_min,
    x_max=workspace_max,
    y_min=workspace_min,
    y_max=workspace_max,
    landmark_positions=landmark_positions,
    num_landmarks=3,
)

motion_model = DoubleIntegratorModel(p_x=5.0, p_y=5.0, v_x=0.0, v_y=0.0, dt=1.0, max_a=2.0)
sensor = RangeBearingSensor(r_max=20.0, epsilon=0.1, sigma_r=0.05, sigma_phi=0.05)

bmdp = BeliefMDP_n_SLAM(
    n=3,
    motion_model=motion_model,
    measurement_model=sensor,
    obstacles=[],
    _map=_map,
    sigma_w=0.01,
    sigma_v=0.01,
    exploration_type="information gain",
)

# Refine observation quantization (n + 2)
bmdp.configure_observation_quantization(5)

# Force recompute Q_n with new observation grid
bmdp.Q_n = bmdp._compute_Q_n()
bmdp._save_Q_n(bmdp._get_Q_cache_path())

print("Q_n shape:", bmdp.Q_n.shape)
print("Observation dim:", bmdp.Y_n.shape[1])

In [ ]:
# Basic stochastic checks on Q_n
Q_n = bmdp.Q_n
m_y, m_n, len_M = Q_n.shape

min_val = float(np.min(Q_n))
max_val = float(np.max(Q_n))
row_sums = np.sum(Q_n, axis=0)  # (m_n, len_M)
max_sum_error = float(np.max(np.abs(row_sums - 1.0)))

print("Q_n min:", min_val)
print("Q_n max:", max_val)
print("Max |sum_y Q_n - 1|:", max_sum_error)

# Inspect a single state-map observation distribution
state_idx = 0
map_idx = 0
obs_dist = Q_n[:, state_idx, map_idx]
print("Obs dist sum:", float(np.sum(obs_dist)))